In [2]:
import pandas as pd
from shopsense.data_generator import generate_customers, generate_transactions, generate_events
from shopsense.features import compute_rfm_features, compute_behavioral_features, compute_transaction_features, build_master_feature_table
from shopsense.models.churn_model import build_churn_preprocessing_pipeline, train_churn_model
from shopsense.evaluation import compute_shap_values, explain_single_prediction, estimate_churn_business_impact

# 1. Quick Data Generation (Standalone setup)
df_c = generate_customers(2000, random_state=42)
df_t = generate_transactions(df_c, random_state=42)
df_e = generate_events(df_c, random_state=42)

# 2. Quick Feature Engineering
snapshot_date = '2023-12-31'
master_df = build_master_feature_table(
    df_c, 
    compute_rfm_features(df_t, snapshot_date), 
    compute_behavioral_features(df_e, df_t, snapshot_date), 
    compute_transaction_features(df_t, snapshot_date)
)

# 3. Setup Preprocessor & Train Model
target = 'churn_label'
categorical_features = ['gender', 'city', 'acquisition_channel', 'rfm_segment', 'preferred_device', 'preferred_category', 'preferred_payment']
numeric_features = [col for col in master_df.columns if col not in categorical_features + [target]]

X = master_df.drop(columns=[target])
y = master_df[target]

preprocessor = build_churn_preprocessing_pipeline(numeric_features, categorical_features)
churn_model = train_churn_model(X, y, preprocessor)

# 4. Transform X for SHAP Explainability
X_transformed = pd.DataFrame(preprocessor.transform(X), columns=preprocessor.get_feature_names_out(), index=X.index)

print("--- SHAP Global Feature Importance ---")
shap_results = compute_shap_values(churn_model, X_transformed)
display(shap_results['mean_abs_shap'].head(10))

print("\n--- Business Impact Estimation ---")
impact = estimate_churn_business_impact(df_c, df_t, churn_model, X)
print(f"Total Revenue at Risk: ${impact['total_revenue_at_risk']:,.2f}")
print(f"Potential Revenue Saved (30% retention): ${impact['potential_save_revenue_30pct']:,.2f}")

2026/06/06 09:10:38 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '216080a1817049ebbfd21b33e370d3f4', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/06/06 09:10:38 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\surya\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Inte

--- SHAP Global Feature Importance ---


num__total_sessions         7.361107
num__age                    0.000000
num__recency_days           0.000000
num__frequency              0.000000
num__monetary_total         0.000000
num__monetary_avg           0.000000
num__rfm_recency_score      0.000000
num__rfm_frequency_score    0.000000
num__rfm_monetary_score     0.000000
num__is_premium             0.000000
dtype: float32


--- Business Impact Estimation ---


2026/06/06 09:10:41 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\surya\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Total Revenue at Risk: $778,433.37
Potential Revenue Saved (30% retention): $233,530.01
